In [1]:
# Task 3: WordNet Semantic Similarity

import nltk
import pandas as pd

# Download required NLTK resources
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.corpus import wordnet as wn

print("WordNet loaded successfully!")

WordNet loaded successfully!


[nltk_data] Downloading package wordnet to /home/parth/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/parth/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
# Select eight word pairs for semantic similarity analysis

word_pairs = [
    ("car", "automobile"),
    ("dog", "cat"),
    ("dog", "animal"),
    ("dog", "computer"),
    ("car", "bicycle"),
    ("teacher", "student"),
    ("apple", "orange"),
    ("book", "airplane")
]

print("Selected word pairs:")
for word1, word2 in word_pairs:
    print(f"- {word1} - {word2}")

print(f"\nTotal pairs: {len(word_pairs)}")

Selected word pairs:
- car - automobile
- dog - cat
- dog - animal
- dog - computer
- car - bicycle
- teacher - student
- apple - orange
- book - airplane

Total pairs: 8


In [3]:
# Select the first noun synset for each word

def get_noun_synset(word):
    synsets = wn.synsets(word, pos=wn.NOUN)
    
    if synsets:
        return synsets[0]
    
    return None


for word1, word2 in word_pairs:
    synset1 = get_noun_synset(word1)
    synset2 = get_noun_synset(word2)
    
    print(f"\n{word1} -> {synset1.name() if synset1 else 'No synset found'}")
    print(f"{word2} -> {synset2.name() if synset2 else 'No synset found'}")


car -> car.n.01
automobile -> car.n.01

dog -> dog.n.01
cat -> cat.n.01

dog -> dog.n.01
animal -> animal.n.01

dog -> dog.n.01
computer -> computer.n.01

car -> car.n.01
bicycle -> bicycle.n.01

teacher -> teacher.n.01
student -> student.n.01

apple -> apple.n.01
orange -> orange.n.01

book -> book.n.01
airplane -> airplane.n.01


In [4]:
# Calculate WordNet path similarity for each word pair

similarity_results = []

for word1, word2 in word_pairs:
    synset1 = get_noun_synset(word1)
    synset2 = get_noun_synset(word2)
    
    if synset1 and synset2:
        similarity = synset1.path_similarity(synset2)
    else:
        similarity = None
    
    similarity_results.append({
        "Word 1": word1,
        "Word 2": word2,
        "Synset 1": synset1.name() if synset1 else "Not found",
        "Synset 2": synset2.name() if synset2 else "Not found",
        "Path Similarity": similarity
    })

similarity_df = pd.DataFrame(similarity_results)

display(similarity_df)

,Word 1,Word 2,Synset 1,Synset 2,Path Similarity
0,car,automobile,car.n.01,car.n.01,1.000000
1,dog,cat,dog.n.01,cat.n.01,0.200000
2,dog,animal,dog.n.01,animal.n.01,0.333333
3,dog,computer,dog.n.01,computer.n.01,0.090909
4,car,bicycle,car.n.01,bicycle.n.01,0.200000
5,teacher,student,teacher.n.01,student.n.01,0.142857
6,apple,orange,apple.n.01,orange.n.01,0.250000
7,book,airplane,book.n.01,airplane.n.01,0.076923


In [5]:
# Display similarity scores for each pair

print("WordNet Path Similarity Scores")
print("=" * 60)

for _, row in similarity_df.iterrows():
    score = row["Path Similarity"]
    
    if pd.notna(score):
        print(f"{row['Word 1']:12} - {row['Word 2']:12} : {score:.4f}")
    else:
        print(f"{row['Word 1']:12} - {row['Word 2']:12} : No score")

WordNet Path Similarity Scores
car          - automobile   : 1.0000
dog          - cat          : 0.2000
dog          - animal       : 0.3333
dog          - computer     : 0.0909
car          - bicycle      : 0.2000
teacher      - student      : 0.1429
apple        - orange       : 0.2500
book         - airplane     : 0.0769


In [6]:
# Rank word pairs according to their semantic similarity

ranked_df = similarity_df.sort_values(
    by="Path Similarity",
    ascending=False,
    na_position="last"
).reset_index(drop=True)

ranked_df.insert(0, "Rank", range(1, len(ranked_df) + 1))

print("Ranked Word Pairs by Semantic Similarity:")
display(ranked_df)

Ranked Word Pairs by Semantic Similarity:


,Rank,Word 1,Word 2,Synset 1,Synset 2,Path Similarity
0,1,car,automobile,car.n.01,car.n.01,1.000000
1,2,dog,animal,dog.n.01,animal.n.01,0.333333
2,3,apple,orange,apple.n.01,orange.n.01,0.250000
3,4,dog,cat,dog.n.01,cat.n.01,0.200000
4,5,car,bicycle,car.n.01,bicycle.n.01,0.200000
5,6,teacher,student,teacher.n.01,student.n.01,0.142857
6,7,dog,computer,dog.n.01,computer.n.01,0.090909
7,8,book,airplane,book.n.01,airplane.n.01,0.076923


In [7]:
# Display the four word pairs specifically mentioned in the assignment

required_pairs = [
    ("car", "automobile"),
    ("dog", "cat"),
    ("dog", "animal"),
    ("dog", "computer")
]

required_df = ranked_df[
    ranked_df.apply(
        lambda row: (row["Word 1"], row["Word 2"]) in required_pairs,
        axis=1
    )
]

print("Required Word Pair Comparisons:")
display(required_df)

Required Word Pair Comparisons:


,Rank,Word 1,Word 2,Synset 1,Synset 2,Path Similarity
0,1,car,automobile,car.n.01,car.n.01,1.000000
1,2,dog,animal,dog.n.01,animal.n.01,0.333333
3,4,dog,cat,dog.n.01,cat.n.01,0.200000
6,7,dog,computer,dog.n.01,computer.n.01,0.090909


In [8]:
# Generate a simple interpretation of the similarity scores

for _, row in ranked_df.iterrows():
    word1 = row["Word 1"]
    word2 = row["Word 2"]
    score = row["Path Similarity"]
    
    if pd.isna(score):
        print(f"{word1} - {word2}: Similarity could not be calculated.")
    elif score >= 0.5:
        print(f"{word1} - {word2}: Relatively high semantic similarity ({score:.4f})")
    elif score >= 0.2:
        print(f"{word1} - {word2}: Moderate semantic similarity ({score:.4f})")
    else:
        print(f"{word1} - {word2}: Relatively low semantic similarity ({score:.4f})")

car - automobile: Relatively high semantic similarity (1.0000)
dog - animal: Moderate semantic similarity (0.3333)
apple - orange: Moderate semantic similarity (0.2500)
dog - cat: Moderate semantic similarity (0.2000)
car - bicycle: Moderate semantic similarity (0.2000)
teacher - student: Relatively low semantic similarity (0.1429)
dog - computer: Relatively low semantic similarity (0.0909)
book - airplane: Relatively low semantic similarity (0.0769)


## Observations

WordNet path similarity measures the semantic relatedness between two
selected synsets based on their position in the WordNet hierarchy.

- **Car and automobile** are expected to have high similarity because they
  represent closely related concepts.
- **Dog and cat** are semantically related because both belong to similar
  categories of animals.
- **Dog and animal** are related through a direct hierarchical relationship,
  since a dog is a type of animal.
- **Dog and computer** are expected to have low similarity because they belong
  to very different semantic categories.

The similarity values differ because WordNet calculates relationships based
on the paths between synsets in its semantic hierarchy. Words that are
closer together in the hierarchy generally receive higher path similarity
than words that are farther apart.